# 03-01 Correlation

Runs the lead-lag **correlation analysis** with Spark: for every ordered pair of tickers, correlate the leader's return today with the follower's return `lag_days` in the future, market-adjust the returns, rank pairs by significance with Benjamini-Hochberg FDR correction, and upload the highest-quality pairs to `strategies/correlation/<run>/data.parquet`.

In [ ]:
# ============================================================================
# SETUP -- installs, imports, config (env vars / config.json -- never hardcoded)
# ============================================================================

# --- Install packages (no-op if already present) --------------------------
# !pip install -q duckdb scipy --upgrade
import os
import json
import sys
import time
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import boto3
import duckdb
# --- Configuration --------------------------------------------------------
# Secrets resolve in priority order:
#   1. Environment variables (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY,
#      AWS_REGION, S3_BUCKET, MASSIVE_API_KEY, ...)
#   2. config.json in the current directory (see config.example.json)
#   3. Built-in defaults (non-secret values only)
# On Kaggle: set secrets via notebook settings (Add-ons -> Secrets), which
# are injected as environment variables.
CONFIG_FILE = "config.json"


def get_secret(name, default=""):
    val = os.environ.get(name)
    if val:
        return val
    if Path(CONFIG_FILE).exists():
        try:
            with open(CONFIG_FILE) as f:
                data = json.load(f)
            if name in data:
                return str(data[name])
        except (OSError, ValueError):
            pass
    return default


class Config:
    def __init__(self):
        self.aws_access_key_id = ""
        self.aws_secret_access_key = ""
        self.aws_region = "us-east-1"
        self.s3_bucket = "market-data-zw"
        self.massive_api_key = ""

        # Paths (S3 keys under the bucket)
        self.types_prefix = "parquet_data/types"
        self.tickers_prefix = "parquet_data/summary/tickers"
        self.ticker_details_prefix = "parquet_data/summary/ticker_yahoo_details"
        self.minute_staging_prefix = "parquet_data/minute_data_staging"
        self.minute_final_prefix = "parquet_data/minute_data_final"
        self.minute_summary_prefix = "parquet_data/summary/minute_summary"
        self.daily_volume_prefix = "parquet_data/summary/daily_volume"
        self.correlation_prefix = "parquet_data/strategies/correlation"
        self.backtest_prefix = "parquet_data/backtest"
        self.backtest_metrics_prefix = "parquet_data/analysis/backtest_metrics"

        # Spark
        self.spark_executor_memory = "24g"
        self.spark_executor_cores = 4
        self.spark_driver_memory = "24g"
        self.spark_tmp = "/tmp/spark"


def load_config():
    cfg = Config()
    if Path(CONFIG_FILE).exists():
        try:
            with open(CONFIG_FILE) as f:
                data = json.load(f)
            for key, value in data.items():
                if hasattr(cfg, key):
                    setattr(cfg, key, value)
        except (OSError, ValueError) as e:
            print(f"[config] WARNING: could not load {CONFIG_FILE}: {e}")

    env_map = {
        "AWS_ACCESS_KEY_ID": "aws_access_key_id",
        "AWS_SECRET_ACCESS_KEY": "aws_secret_access_key",
        "AWS_REGION": "aws_region",
        "S3_BUCKET": "s3_bucket",
        "MASSIVE_API_KEY": "massive_api_key",
        "SPARK_DRIVER_MEMORY": "spark_driver_memory",
        "SPARK_EXECUTOR_MEMORY": "spark_executor_memory",
        "SPARK_EXECUTOR_CORES": "spark_executor_cores",
    }
    for env_name, attr in env_map.items():
        val = os.environ.get(env_name)
        if val:
            if attr == "spark_executor_cores":
                val = int(val)
            setattr(cfg, attr, val)
    return cfg
# --- S3 helpers -----------------------------------------------------------
def s3_client(cfg):
    from botocore.config import Config as BotocoreConfig
    config = BotocoreConfig(retries={"max_attempts": 5, "mode": "adaptive"},
                            connect_timeout=30, read_timeout=60)
    return boto3.client("s3",
                        aws_access_key_id=cfg.aws_access_key_id,
                        aws_secret_access_key=cfg.aws_secret_access_key,
                        region_name=cfg.aws_region,
                        config=config)


def upload_parquet(df, s3, bucket, key, compression="snappy"):
    buf = BytesIO()
    df.to_parquet(buf, index=False, engine="pyarrow", compression=compression,
                  coerce_timestamps="ms", allow_truncated_timestamps=True)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())


def download_parquet(s3, bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_parquet(BytesIO(obj["Body"].read()))


def list_s3_keys(s3, bucket, prefix):
    keys = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            keys.append(obj["Key"])
    return keys


def tickers_from_prefix(s3, bucket, prefix):
    """Ticker symbols from `<prefix>/<TICKER>.parquet` object keys."""
    return [k.split("/")[-1][:-len(".parquet")] for k in list_s3_keys(s3, bucket, prefix)
            if k.endswith(".parquet")]


def duckdb_s3_connect(cfg):
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute(f"SET s3_access_key_id='{cfg.aws_access_key_id}';")
    con.execute(f"SET s3_secret_access_key='{cfg.aws_secret_access_key}';")
    con.execute(f"SET s3_region='{cfg.aws_region}';")
    return con


def spark_session(cfg):
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder
        .appName("MarketDataPlatform")
        .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.1")
        .config("spark.executor.memory", cfg.spark_executor_memory)
        .config("spark.executor.cores", str(cfg.spark_executor_cores))
        .config("spark.driver.memory", cfg.spark_driver_memory)
        .config("spark.hadoop.fs.s3a.access.key", cfg.aws_access_key_id)
        .config("spark.hadoop.fs.s3a.secret.key", cfg.aws_secret_access_key)
        .config("spark.hadoop.fs.s3a.endpoint", f"s3.{cfg.aws_region}.amazonaws.com")
        .config("spark.local.dir", cfg.spark_tmp)
        .config("spark.hadoop.tmp.dir", cfg.spark_tmp)
        .config("spark.sql.warehouse.dir", f"{cfg.spark_tmp}/warehouse")
        .getOrCreate()
    )
    spark.conf.set("spark.hadoop.fs.s3a.committer.name", "directory")
    spark.conf.set("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")
    spark.conf.set("spark.hadoop.fs.s3a.committer.staging.conflict-mode", "append")
    spark.conf.set("spark.sql.debug.maxToStringFields", "100")
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
    spark.conf.set("spark.sql.ansi.enabled", "false")
    spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
    spark.conf.set("spark.sql.parquet.mergeSchema", "true")
    spark.sparkContext.setLogLevel("ERROR")
    return spark

os.makedirs("/tmp/spark", exist_ok=True)

# --- Instantiate config + clients -----------------------------
cfg = load_config()
s3 = s3_client(cfg)
spark = spark_session(cfg)
print("Setup complete")
print(f"Bucket: {cfg.s3_bucket} | Region: {cfg.aws_region}")


In [ ]:
# ============================================================================
# Data source resolution -- prefer the local Kaggle dataset mirror (created
# by 02-02-s3-to-kaggle-dataset: fast, free reads), fall back to S3.
# ============================================================================

import glob as _glob

MIRROR_CANDIDATES = [
    "/kaggle/input/datasets/dsptlp/market-data-s3-dataset/s3_data/parquet_data",
    "/kaggle/input/market-data-s3-dataset/s3_data/parquet_data",
]


def resolve(rel, name="", use_glob=False):
    """Kaggle-mirror path when mounted, else s3a:// URI."""
    short = rel[len("parquet_data/"):] if rel.startswith("parquet_data/") else rel
    for root in MIRROR_CANDIDATES:
        local = os.path.join(root, short, name)
        if use_glob:
            if _glob.glob(local):
                return local
        elif os.path.exists(local):
            return local
    return f"s3a://{cfg.s3_bucket}/{rel}/{name}"


In [ ]:
# ============================================================================
# Correlation engine (filters + features + correlation) -- self-contained helpers (no external package imports)
# ============================================================================


from __future__ import annotations

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

# =============================================================================
# Core filter functions
# =============================================================================

def restrict_date_range(
    df: DataFrame,
    start_date: str,
    end_date: str,
    date_col: str = "date",
) -> DataFrame:
    """Filter to an inclusive date range."""
    return df.filter(
        (F.col(date_col) >= F.lit(start_date)) &
        (F.col(date_col) <= F.lit(end_date))
    )


def filter_active_tickers(
    df: DataFrame,
    min_rows: int = 150,
    ticker_col: str = "ticker",
) -> DataFrame:
    """Keep tickers with at least ``min_rows`` observations."""
    active = (
        df.groupBy(ticker_col)
        .count()
        .filter(F.col("count") >= min_rows)
        .select(ticker_col)
    )
    return df.join(active, ticker_col, "inner")


def limit_tickers(
    df: DataFrame,
    max_tickers: int = 300,
    ticker_col: str = "ticker",
    seed: int | None = None,
) -> DataFrame:
    """Randomly sample up to ``max_tickers`` tickers."""
    import time

    if seed is None:
        seed = int(time.time() * 1_000_000) % 2_147_483_647
    tickers = (
        df.select(ticker_col)
        .distinct()
        .withColumn("_rand", F.rand(seed))
        .orderBy("_rand")
        .limit(max_tickers)
        .drop("_rand")
    )
    return df.join(tickers, ticker_col, "inner")


def filter_regular_trading(
    df: DataFrame,
    max_gap_days: int = 5,
    ticker_col: str = "ticker",
    date_col: str = "date",
) -> DataFrame:
    """Keep tickers with no trading gap larger than ``max_gap_days``."""
    w = Window.partitionBy(ticker_col).orderBy(date_col)
    gaps = (
        df.withColumn("gap", F.datediff(F.col(date_col), F.lag(date_col).over(w)))
        .groupBy(ticker_col)
        .agg(F.max("gap").alias("max_gap"))
        .filter(F.col("max_gap") <= max_gap_days)
        .select(ticker_col)
    )
    return df.join(gaps, ticker_col, "inner")


def filter_extreme_volatility(
    df: DataFrame,
    max_daily_vol: float = 0.15,
    ticker_col: str = "ticker",
    price_col: str = "price",
) -> DataFrame:
    """Remove tickers whose price volatility (stddev) exceeds the ceiling."""
    vol = (
        df.groupBy(ticker_col)
        .agg(F.stddev(price_col).alias("vol"))
        .filter(F.col("vol") <= max_daily_vol)
        .select(ticker_col)
    )
    return df.join(vol, ticker_col, "inner")


def filter_calendar_coverage(
    df: DataFrame,
    min_coverage_ratio: float = 0.9,
    ticker_col: str = "ticker",
    date_col: str = "date",
) -> DataFrame:
    """Keep tickers trading on at least ``min_coverage_ratio`` of all dates."""
    total_days = df.select(date_col).distinct().count()
    coverage = (
        df.groupBy(ticker_col)
        .agg(F.countDistinct(date_col).alias("days"))
        .withColumn("coverage", F.col("days") / total_days)
        .filter(F.col("coverage") >= min_coverage_ratio)
        .select(ticker_col)
    )
    return df.join(coverage, ticker_col, "inner")


def filter_liquid_tickers(
    df: DataFrame,
    min_avg_volume: float = 5,
    ticker_col: str = "ticker",
    volume_col: str = "volume",
) -> DataFrame:
    """Keep tickers with minimum average volume."""
    liquid = (
        df.groupBy(ticker_col)
        .agg(F.avg(volume_col).alias("avg_vol"))
        .filter(F.col("avg_vol") >= min_avg_volume)
        .select(ticker_col)
    )
    return df.join(liquid, ticker_col, "inner")


def remove_tickers_with_zero_or_null_price(
    df: DataFrame,
    price_col: str = "price",
    ticker_col: str = "ticker",
) -> DataFrame:
    """Remove tickers with any zero or null price."""
    valid = (
        df.groupBy(ticker_col)
        .agg(F.min(price_col).alias("min_price"))
        .filter(F.col("min_price").isNotNull() & (F.col("min_price") != 0))
        .select(ticker_col)
    )
    return df.join(valid, on=ticker_col, how="inner")


# =============================================================================
# Composite filter pipelines
# =============================================================================

def apply_quality_filters(
    df: DataFrame,
    start_date: str = "2024-01-01",
    end_date: str = "2026-01-01",
    min_rows: int = 150,
    min_coverage_ratio: float = 0.9,
    max_gap_days: int = 60,
    min_avg_volume: float = 5,
    max_daily_vol: float = 0.9,
    track: bool = False,
) -> DataFrame:
    """
    Apply the deterministic quality filters (date range -> volatility).

    Does NOT include the random ticker-limit step, so results are
    reproducible across runs.
    """
    steps = [
        ("Date Range", lambda d: restrict_date_range(d, start_date, end_date)),
        ("Min Observations", lambda d: filter_active_tickers(d, min_rows=min_rows)),
        ("Price Validity", lambda d: remove_tickers_with_zero_or_null_price(d)),
        ("Calendar Coverage", lambda d: filter_calendar_coverage(
            d, min_coverage_ratio=min_coverage_ratio)),
        ("Regular Trading", lambda d: filter_regular_trading(d, max_gap_days=max_gap_days)),
        ("Liquidity", lambda d: filter_liquid_tickers(d, min_avg_volume=min_avg_volume)),
        ("Volatility", lambda d: filter_extreme_volatility(d, max_daily_vol=max_daily_vol)),
    ]
    return _run_steps(df, steps, track)


def apply_all_filters(
    df: DataFrame,
    start_date: str = "2024-01-01",
    end_date: str = "2026-01-01",
    min_rows: int = 150,
    max_tickers: int = 1000,
    min_coverage_ratio: float = 0.9,
    max_gap_days: int = 60,
    min_avg_volume: float = 5,
    max_daily_vol: float = 0.9,
    seed: int | None = None,
    track: bool = False,
) -> DataFrame:
    """
    Apply all quality filters in standard order, with the random ticker
    limit applied LAST (so it samples the fully-filtered universe).
    """
    steps = [
        ("Date Range", lambda d: restrict_date_range(d, start_date, end_date)),
        ("Min Observations", lambda d: filter_active_tickers(d, min_rows=min_rows)),
        ("Price Validity", lambda d: remove_tickers_with_zero_or_null_price(d)),
        ("Calendar Coverage", lambda d: filter_calendar_coverage(
            d, min_coverage_ratio=min_coverage_ratio)),
        ("Regular Trading", lambda d: filter_regular_trading(d, max_gap_days=max_gap_days)),
        ("Liquidity", lambda d: filter_liquid_tickers(d, min_avg_volume=min_avg_volume)),
        ("Volatility", lambda d: filter_extreme_volatility(d, max_daily_vol=max_daily_vol)),
        ("Ticker Limit", lambda d: limit_tickers(d, max_tickers=max_tickers, seed=seed)),
    ]
    return _run_steps(df, steps, track)


def _run_steps(df: DataFrame, steps, track: bool) -> DataFrame:
    prev_rows = prev_tickers = None
    if track:
        prev_rows, prev_tickers = get_stats(df)
        print(f"\n{'Step':<30} {'Rows':>15} {'Tickers':>10} "
              f"{'Rows Removed':>15} {'Tickers Removed':>15}")
        print("-" * 85)
        print(f"{'0. Starting Data':<30} {prev_rows:>15,} {prev_tickers:>10,} "
              f"{'-':>15} {'-':>15}")

    for name, fn in steps:
        df = df.transform(fn)
        if track:
            rows, tickers = get_stats(df)
            print_filter_stats(name, prev_rows, prev_tickers, rows, tickers)
            prev_rows, prev_tickers = rows, tickers

    return df


# =============================================================================
# Stats helpers
# =============================================================================

def get_stats(df: DataFrame, label: str = ""):
    """Return (row_count, ticker_count) for a DataFrame."""
    rows = df.count()
    tickers = df.select("ticker").distinct().count() if "ticker" in df.columns else 0
    return rows, tickers


def print_filter_stats(step_name, prev_rows, prev_tickers, curr_rows, curr_tickers):
    """Print a standardized per-filter stats row."""
    print(f"{step_name:<30} {curr_rows:>15,} {curr_tickers:>10,} "
          f"{prev_rows - curr_rows:>15,} {prev_tickers - curr_tickers:>15,}")



from __future__ import annotations

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F


def prepare_features(
    df: DataFrame,
    lag_days: int = 7,
    lookback_days: int = 7,
) -> DataFrame:
    """
    Calculate returns, lagged features, lead features and rolling averages.

    Parameters
    ----------
    df : DataFrame
        Input with columns: ``ticker``, ``price``, ``date``, ``volume``.
    lag_days : int
        Prediction horizon in days (lead window).
    lookback_days : int
        Rolling-average window (rows before current row).

    Returns
    -------
    DataFrame with additional columns:

    - ``return``: daily return
    - ``price_lag_{lag_days}``: price ``lag_days`` ago
    - ``return_lag_{lag_days}``: return ``lag_days`` ago
    - ``return_lead_{lag_days}``: return ``lag_days`` in the FUTURE (target)
    - ``follower_window_return_{lag_days}d``: cumulative return over the lag
    - ``pre_move_avg_return``: average return over the lookback window
    """
    w = Window.partitionBy("ticker").orderBy("date")
    w_rolling = Window.partitionBy("ticker").orderBy("date").rowsBetween(-lookback_days, -1)

    return (
        df
        .withColumn("prev_price", F.lag("price").over(w))
        .withColumn("return", (F.col("price") - F.col("prev_price")) / F.col("prev_price"))
        .filter(F.col("return").isNotNull())

        # Historical (lagged) features
        .withColumn(f"price_lag_{lag_days}", F.lag("price", lag_days).over(w))
        .withColumn(f"return_lag_{lag_days}", F.lag("return", lag_days).over(w))

        # Future (lead) feature -- the prediction target
        .withColumn(f"return_lead_{lag_days}", F.lead("return", lag_days).over(w))

        # Cumulative return over the lag window
        .withColumn(
            f"follower_window_return_{lag_days}d",
            (F.col("price") - F.col(f"price_lag_{lag_days}"))
            / F.col(f"price_lag_{lag_days}"),
        )

        # Momentum indicator: rolling average return
        .withColumn("pre_move_avg_return", F.avg("return").over(w_rolling))

        .drop("prev_price")
    )



from __future__ import annotations

import time
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql.types import *  # noqa: F401,F403
from scipy import stats


DEFAULT_EXCLUDED_TICKER_TYPES = ["ETF", "ETN", "ETV", "ETS"]


# =============================================================================
# Pair analysis
# =============================================================================

def analyze_pairs(
    df: DataFrame,
    lag_days: int = 7,
    window_days: int = 30,
    sector_df: DataFrame = None,
    market_df: DataFrame = None,
    adjust_returns: bool = False,
    debug: bool = False,
) -> DataFrame:
    """
    Compute aggregate and rolling correlations for all stock pairs.

    Parameters
    ----------
    df : DataFrame
        Feature-enriched DataFrame from :func:`prepare_features`.
    lag_days : int
        Lag (prediction horizon) in days.
    window_days : int
        Rolling window size for correlation.
    sector_df : DataFrame
        Optional ``ticker`` / ``sector`` mapping; same-sector pairs excluded.
    market_df : DataFrame
        Optional ``date`` / ``market_return`` mapping for market adjustment.
    adjust_returns : bool
        Subtract the market return before correlating.
    debug : bool
        Print per-stage row diagnostics.

    Returns
    -------
    DataFrame with one row per (leader, follower) per rolling window.
    """
    a = df.alias("a")
    b = df.alias("b")

    joined = a.join(
        b,
        (F.col("a.date") == F.col("b.date")) & (F.col("a.ticker") < F.col("b.ticker")),
        "inner",
    )

    if sector_df is not None:
        joined = (
            joined
            .join(sector_df.alias("a_sector"), F.col("a.ticker") == F.col("a_sector.ticker"), "left")
            .join(sector_df.alias("b_sector"), F.col("b.ticker") == F.col("b_sector.ticker"), "left")
            .filter(
                (F.col("a_sector.sector").isNull())
                | (F.col("b_sector.sector").isNull())
                | (F.col("a_sector.sector") != F.col("b_sector.sector"))
            )
        )

    if market_df is not None and adjust_returns:
        joined = (
            joined.join(market_df.alias("mkt"), on="date", how="left")
            .withColumn("mkt_return", F.col("mkt.market_return"))
            .withColumn("a_return_adj", F.col("a.return") - F.col("mkt_return"))
            .withColumn(
                f"b_return_lead_adj_{lag_days}",
                F.col(f"b.return_lead_{lag_days}") - F.col("mkt_return"),
            )
        )
    elif adjust_returns and market_df is None:
        adjust_returns = False

    if adjust_returns:
        corr_a = F.col("a_return_adj")
        corr_b = F.col(f"b_return_lead_adj_{lag_days}")
    else:
        corr_a = F.col("a.return")
        corr_b = F.col(f"b.return_lead_{lag_days}")

    if debug:
        _d = joined.agg(
            F.count("*").alias("rows"),
            F.sum(corr_a.isNotNull().cast("long")).alias("a_nonnull"),
            F.sum(corr_b.isNotNull().cast("long")).alias("b_nonnull"),
        ).first()
        print(f"   [debug] analyze_pairs self-join: rows={_d.rows:,} "
              f"corr_a_nonnull={_d.a_nonnull:,} corr_b_nonnull={_d.b_nonnull:,}")

    # Aggregate statistics (one value per pair across all time)
    pair_stats = (
        joined
        .groupBy(
            F.col("a.ticker").alias("leader"),
            F.col("b.ticker").alias("follower"),
        )
        .agg(
            F.corr(corr_a, corr_b).alias("correlation"),
            F.avg(F.signum(corr_a * corr_b)).alias("sign_fraction"),
            F.avg("b.pre_move_avg_return").alias(f"avg_pre_{lag_days}d_return"),
            F.avg(F.col(f"b.follower_window_return_{lag_days}d"))
            .alias(f"avg_follower_{lag_days}d_price_change"),
        )
    )

    # Rolling correlation window (changes over time)
    w = (
        Window.partitionBy("a.ticker", "b.ticker")
        .orderBy(F.col("a.date").cast("long"))
        .rowsBetween(-window_days + 1, 0)
    )

    rolling = (
        joined
        .withColumn("rolling_corr", F.corr(corr_a, corr_b).over(w))
        .withColumn("window_start_date", F.min(F.col("a.date")).over(w))
        .withColumn("window_end_date", F.max(F.col("a.date")).over(w))
        .withColumn(
            "follower_window_return",
            (F.last("b.price").over(w) - F.first("b.price").over(w))
            / F.first("b.price").over(w),
        )
        .select(
            F.col("a.ticker").alias("leader"),
            F.col("b.ticker").alias("follower"),
            F.col("a.date").alias("leader_date"),
            F.col("a.price").alias("leader_price"),
            F.col("a.volume").alias("leader_volume"),
            F.col("a.return").alias("leader_return"),
            F.col(f"a.price_lag_{lag_days}").alias(f"leader_price_lag_{lag_days}"),
            F.col(f"a.return_lag_{lag_days}").alias(f"leader_return_lag_{lag_days}"),
            F.col(f"a.follower_window_return_{lag_days}d")
            .alias(f"leader_follower_window_return_{lag_days}d"),
            F.col("a.pre_move_avg_return").alias("leader_pre_move_avg_return"),
            F.col("b.date").alias("follower_date"),
            F.col("b.price").alias("follower_price"),
            F.col("b.volume").alias("follower_volume"),
            F.col("b.return").alias("follower_return"),
            F.col(f"b.price_lag_{lag_days}").alias(f"follower_price_lag_{lag_days}"),
            F.col(f"b.return_lag_{lag_days}").alias(f"follower_return_lag_{lag_days}"),
            F.col(f"b.return_lead_{lag_days}").alias(f"follower_return_lead_{lag_days}"),
            F.col(f"b.follower_window_return_{lag_days}d")
            .alias(f"follower_follower_window_return_{lag_days}d"),
            F.col("b.pre_move_avg_return").alias("follower_pre_move_avg_return"),
            "rolling_corr",
            "window_start_date",
            "window_end_date",
            "follower_window_return",
        )
    )

    return rolling.join(pair_stats, ["leader", "follower"], "left")


def run_lagged_stock_search(
    df: DataFrame,
    lag_days: int = 7,
    lookback_days: int = 7,
    persistence_window_days: int = 10,
    add_significance: bool = False,
    min_significance: float = 0.3,
    fdr_q: float = 0.10,
    sector_df: DataFrame = None,
    market_df: DataFrame = None,
    adjust_returns: bool = False,
    debug: bool = False,
) -> DataFrame:
    """
    End-to-end pipeline for lead-lag stock pair discovery.

    Parameters
    ----------
    df : DataFrame
        Input with columns: ``ticker``, ``price``, ``date``, ``volume``.
    lag_days : int
        Prediction horizon in days.
    lookback_days : int
        Days for rolling-average calculations.
    persistence_window_days : int
        Window size for rolling correlation.
    add_significance : bool
        Add ``significance_score`` / ``significance_rank`` columns.
    min_significance : float
        Significance threshold used to build the ``passes_significance`` column.
    fdr_q : Optional[float]
        Benjamini-Hochberg FDR threshold; ``None`` disables.
    sector_df, market_df, adjust_returns, debug
        Forwarded to :func:`analyze_pairs`.

    Returns
    -------
    DataFrame with lead-lag pair relationships and statistics.
    """
    enriched = prepare_features(df, lag_days, lookback_days)
    if debug:
        print(f"   [debug] run_lagged_stock_search: prepare_features rows={enriched.count():,}")

    results = analyze_pairs(
        enriched,
        lag_days,
        persistence_window_days,
        sector_df,
        market_df,
        adjust_returns,
        debug=debug,
    )

    if add_significance:
        w = Window.orderBy(F.desc("significance_score"))
        results = (
            results
            .withColumn("significance_score", F.abs("correlation"))
            .withColumn("significance_rank", F.row_number().over(w))
            .withColumn("significance_total_tests", F.count("*").over(Window.partitionBy()))
            .withColumn("significance_threshold", F.lit(min_significance))
            .withColumn(
                "passes_significance",
                F.col("significance_score") >= F.col("significance_threshold"),
            )
        )

    if fdr_q is not None and fdr_q > 0:
        results = compute_bh_fdr(results, q=fdr_q, lag_days=lag_days)

    return results


# =============================================================================
# Market returns
# =============================================================================

def compute_market_returns(df: DataFrame) -> DataFrame:
    """Daily market return (median % move) across all tickers."""
    ticker_window = Window.partitionBy("ticker").orderBy("date")
    dod = (
        df.select("ticker", "date", "price")
        .withColumn("prev_price", F.lag("price").over(ticker_window))
        .withColumn(
            "dod_pct",
            ((F.col("price") - F.col("prev_price")) / F.col("prev_price")) * 100,
        )
        .filter(F.col("dod_pct").isNotNull())
        .filter(F.col("dod_pct").between(-50, 50))
    )
    return (
        dod.groupBy("date")
        .agg(F.percentile_approx("dod_pct", 0.5).alias("market_return"))
        .orderBy("date")
    )


def compute_market_returns_decimal(df: DataFrame) -> DataFrame:
    """Daily market return as a decimal (e.g. ``0.05``), matching feature scale."""
    return compute_market_returns(df).withColumn("market_return", F.col("market_return") / 100.0)


def calc_market_dod(
    df: DataFrame,
    start_date: str | None = None,
    end_date: str | None = None,
    outlier_bound: float = 50,
) -> pd.DataFrame:
    """
    Calculate day-over-day market return across all tickers (pandas).

    Returns a pandas DataFrame with columns:
    ``date``, ``market_dod_pct`` (median), ``market_dod_mean``, ``tickers_counted``.
    """
    filtered = df
    if start_date:
        filtered = filtered.filter(F.col("date") >= start_date)
    if end_date:
        filtered = filtered.filter(F.col("date") <= end_date)

    ticker_window = Window.partitionBy("ticker").orderBy("date")
    dod = (
        filtered.select("ticker", "date", "price")
        .withColumn("prev_price", F.lag("price").over(ticker_window))
        .withColumn(
            "dod_pct",
            ((F.col("price") - F.col("prev_price")) / F.col("prev_price")) * 100,
        )
        .filter(F.col("dod_pct").isNotNull())
        .filter(F.col("dod_pct").between(-outlier_bound, outlier_bound))
    )
    market_returns = (
        dod.groupBy("date")
        .agg(
            F.percentile_approx("dod_pct", 0.5).alias("market_dod_pct"),
            F.avg("dod_pct").alias("market_dod_mean"),
            F.count("ticker").alias("tickers_counted"),
        )
        .orderBy("date")
        .toPandas()
    )
    return market_returns


# =============================================================================
# FDR correction
# =============================================================================

def compute_bh_fdr(results: DataFrame, q: float = 0.10, lag_days: int = 7) -> DataFrame:
    """
    Apply Benjamini-Hochberg FDR correction to significance scores.

    Converts ``|correlation|`` to p-values via the t-distribution, then
    computes BH-adjusted q-values and a ``passes_fdr`` flag.
    """
    pair_stats = (
        results.groupBy("leader", "follower")
        .agg(
            F.first("correlation").alias("correlation"),
            F.count(
                F.when(
                    F.col("leader_return").isNotNull()
                    & F.col(f"follower_return_lead_{lag_days}").isNotNull(),
                    1,
                )
            ).alias("n_obs"),
        )
        .toPandas()
    )

    if len(pair_stats) == 0:
        return (
            results.withColumn("p_value", F.lit(1.0))
            .withColumn("q_value", F.lit(1.0))
            .withColumn("passes_fdr", F.lit(False))
        )

    n = pair_stats["n_obs"].values
    r = np.clip(pair_stats["correlation"].values, -0.999999, 0.999999)
    mask = n < 2
    n_safe = np.where(mask, 2, n)
    t_stat = r * np.sqrt(n_safe - 2) / np.sqrt(1 - r * r)
    df = np.maximum(n_safe - 2, 1)
    p_values = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=df))
    p_values[mask] = 1.0
    pair_stats["p_value"] = p_values

    m = len(p_values)
    sorted_idx = np.argsort(p_values)
    sorted_p = p_values[sorted_idx]
    ranks = np.arange(1, m + 1)
    q_values = np.empty(m)
    q_values[sorted_idx] = np.minimum.accumulate((sorted_p * m / ranks)[::-1])[::-1]
    q_values = np.clip(q_values, 0.0, 1.0)
    pair_stats["q_value"] = q_values
    pair_stats["passes_fdr"] = q_values <= q

    lookup = results.sparkSession.createDataFrame(
        [
            (row["leader"], row["follower"], float(row["p_value"]),
             float(row["q_value"]), bool(row["passes_fdr"]))
            for _, row in pair_stats.iterrows()
        ],
        ["leader", "follower", "p_value", "q_value", "passes_fdr"],
    )
    return results.join(lookup, ["leader", "follower"], "left")


# =============================================================================
# Pipeline (notebook 03-01)
# =============================================================================

def load_daily_data(
    spark,
    minute_summary_path: str,
    daily_volume_path: str,
    tickers_path: str,
    excluded_types: list[str] | None = None,
) -> DataFrame:
    """
    Load the daily close prices + real daily volume and exclude ticker types.

    Mirrors the ``03-01`` load + ticker-type-exclusion cells. Returns a
    cached DataFrame with columns ``ticker``, ``price``, ``date``, ``volume``,
    ``trades``.
    """
    excluded_types = excluded_types or DEFAULT_EXCLUDED_TICKER_TYPES

    df_raw = spark.read.parquet(minute_summary_path).filter(F.col("rn_desc").isin(1))
    vol = spark.read.parquet(daily_volume_path)

    df = (
        df_raw
        .select(
            F.col("symbol").alias("ticker"),
            F.col("close").alias("price"),
            F.col("trade_date").alias("date"),
        )
        .join(
            vol.select(
                F.col("symbol").alias("ticker"),
                F.col("trade_date").alias("date"),
                F.col("volume").alias("volume"),
                F.col("trades").alias("trades"),
            ),
            ["ticker", "date"],
            "left",
        )
        .cache()
    )

    tickers_df = spark.read.parquet(tickers_path).select("ticker", "type")
    excluded_df = F.broadcast(
        tickers_df
        .filter(F.col("type").isin(excluded_types))
        .select(F.col("ticker").alias("excluded_ticker"))
    )
    df = (
        df.join(excluded_df, df.ticker == F.col("excluded_ticker"), "left")
        .filter(F.col("excluded_ticker").isNull())
        .drop("excluded_ticker")
    )
    return df


def select_high_quality_pairs(
    results: DataFrame,
    min_significance: float,
    min_correlation: float,
    min_sign_fraction: float,
    recent_days: int,
    result_limit: int,
    lag_days: int,
) -> pd.DataFrame:
    """
    Apply the ``03-01`` high-quality pair query and return a pandas DataFrame.

    Columns referenced by the query are derived from ``lag_days``.
    """
    corr_col = "correlation"
    sign_frac_col = "sign_fraction"
    avg_follower_col = f"avg_follower_{lag_days}d_price_change"
    avg_pre_col = f"avg_pre_{lag_days}d_return"
    sig_col = "significance_score"
    rolling_corr_col = "rolling_corr"
    leader_date_col = "leader_date"
    leader_col = "leader"
    follower_col = "follower"

    results.createOrReplaceTempView("stock_pairs")

    max_leader_date = (
        results.select(F.max(F.col(leader_date_col)).alias("max_leader_date"))
        .first()["max_leader_date"]
    )
    if max_leader_date is None:
        raise ValueError("No leader_date values found in results")

    recent_cutoff_date = max_leader_date - timedelta(days=recent_days)

    query = f"""
        SELECT *
        FROM (
            SELECT
                {leader_col},
                {follower_col},
                {corr_col},
                {sign_frac_col},
                {avg_follower_col},
                {avg_pre_col},
                {sig_col},
                ABS({corr_col}) AS abs_correlation,
                ABS({avg_follower_col}) AS abs_expected_move,
                COUNT(*) AS num_observations,
                AVG(
                    CASE
                        WHEN {leader_date_col} >= TIMESTAMP('{recent_cutoff_date}')
                        THEN {rolling_corr_col}
                    END
                ) AS recent_{recent_days}d_correlation,
                MAX({leader_date_col}) AS last_observation_date,
                ROW_NUMBER() OVER (
                    PARTITION BY {leader_col}, {follower_col}
                    ORDER BY ABS({corr_col}) * ABS({sign_frac_col}) DESC
                ) AS rn
            FROM stock_pairs
            WHERE {sig_col} >= {min_significance}
            GROUP BY
                {leader_col}, {follower_col}, {corr_col}, {sign_frac_col},
                {avg_follower_col}, {avg_pre_col}, {sig_col}
            HAVING
                ABS({corr_col}) >= {min_correlation}
                AND ABS({sign_frac_col}) >= {min_sign_fraction}
        )
        WHERE rn = 1
        ORDER BY
            ABS({corr_col}) DESC,
            ABS({sign_frac_col}) DESC,
            ABS({avg_follower_col}) DESC
        LIMIT {result_limit}
    """
    return results.sparkSession.sql(query).toPandas()








In [ ]:
# ============================================================================
# Analysis parameters
# ============================================================================

import random

# --- Randomized lead-lag parameters (new values each run) --------------------
lag_days           = random.randint(1, 20)      # prediction horizon (days)
lookback_days      = random.randint(1, 30)      # rolling-average window (days)
persistence_window = random.randint(5, 30)  # rolling-corr window (days)
print(f"Randomized: lag_days={lag_days}, lookback_days={lookback_days}, persistence_window={persistence_window}")

# --- Data window ------------------------------------------------------------
start_date = "2024-02-01"        # inclusive start
end_date   = "2026-01-01"        # inclusive end

# --- Ticker universe ---------------------------------------------------------
max_tickers = 1000               # random sample size, applied AFTER quality filters

# --- Lead-lag analysis -------------------------------------------------------
add_significance   = True        # add significance score/rank columns
min_significance   = 0.2         # significance_score threshold
adjust_returns     = True        # market-adjust returns before correlation
fdr_q              = 0.10        # Benjamini-Hochberg FDR threshold

# --- Quality filters (applied in order; ticker limit is LAST) ----------------
min_observations   = 150         # min rows per ticker
min_coverage_ratio = 0.9         # calendar coverage ratio
max_gap_days       = 60          # regular-trading max gap (days)
min_avg_volume     = 1000        # liquidity floor (avg daily share volume)
max_daily_vol      = 0.9         # volatility ceiling (sigma)

# --- Ticker type exclusions (ETF, ETN, ETV, ETS) ------------------------------
EXCLUDED_TICKER_TYPES = ["ETF", "ETN", "ETV", "ETS"]

# --- High-quality pair selection (SQL query thresholds) ----------------------
min_correlation   = 0.85         # |correlation| floor (relaxed)
min_sign_fraction = 0.5          # |sign_fraction| floor
recent_days       = 30           # rolling-corr recency window (days)
result_limit      = 500          # max pairs per run

spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
spark.conf.set("spark.sql.debug.maxToStringFields", "100")
print("Parameters set")

In [ ]:
# ============================================================================
# Load daily close prices + real daily volume, exclude ETF/ETN/ETV/ETS
# ============================================================================

MINUTE_SUMMARY = resolve(cfg.minute_summary_prefix, "data.parquet")
DAILY_VOLUME   = resolve(cfg.daily_volume_prefix, "data.parquet")
TICKERS_PATH   = resolve(cfg.tickers_prefix, "tickers.parquet")

# daily close (last minute bar of each day) + real daily volume
df_raw = spark.read.parquet(MINUTE_SUMMARY).filter(F.col("rn_desc").isin(1))
vol = spark.read.parquet(DAILY_VOLUME)

df = (
    df_raw
    .select(
        F.col("symbol").alias("ticker"),
        F.col("close").alias("price"),
        F.col("trade_date").alias("date"),
    )
    .join(
        vol.select(
            F.col("symbol").alias("ticker"),
            F.col("trade_date").alias("date"),
            F.col("volume").alias("volume"),
            F.col("trades").alias("trades"),
        ),
        ["ticker", "date"],
        "left",
    )
    .cache()
)

# Ticker type exclusion (ETF / ETN / ETV / ETS)
tickers_df = spark.read.parquet(TICKERS_PATH).select("ticker", "type")
excluded_df = F.broadcast(
    tickers_df
    .filter(F.col("type").isin(EXCLUDED_TICKER_TYPES))
    .select(F.col("ticker").alias("excluded_ticker"))
)
df = (
    df.join(excluded_df, df.ticker == F.col("excluded_ticker"), "left")
    .filter(F.col("excluded_ticker").isNull())
    .drop("excluded_ticker")
)

before_rows, before_tickers = get_stats(df)
print(f"After type exclusion: {before_rows:,} rows, {before_tickers:,} tickers")

In [ ]:
# ============================================================================
# Apply the deterministic quality filters (once, cached result)
# ============================================================================

df_filtered = apply_quality_filters(
    df,
    start_date=start_date,
    end_date=end_date,
    min_rows=min_observations,
    min_coverage_ratio=min_coverage_ratio,
    max_gap_days=max_gap_days,
    min_avg_volume=min_avg_volume,
    max_daily_vol=max_daily_vol,
    track=True,   # prints the per-filter impact table
).cache()

initial_rows, initial_tickers = get_stats(df)
final_rows, final_tickers = get_stats(df_filtered)

print("-" * 85)
print(f"{'FINAL CLEANED DATASET':<30} {final_rows:>12,} rows ({final_rows/initial_rows*100:.1f}% of original)")
print(f"{'TICKERS':<30} {final_tickers:>12,} ({final_tickers/initial_tickers*100:.1f}% of original)")
print("=" * 70)

In [ ]:
# ============================================================================
# Lead-lag analysis: market returns -> random sample -> pair search
# ============================================================================

import time
from datetime import timedelta

start_time = time.time()

# 1. Market returns (median daily move, decimal form for adjustment)
market_df = compute_market_returns_decimal(df_filtered)
print("   [OK] Computed market returns")

# 2. Fresh random sample of tickers
df_iter = limit_tickers(df_filtered, max_tickers=max_tickers, seed=None)
print(f"   [OK] Ticker sample prepared (max {max_tickers:,})")

# 3. Lead-lag analysis over all ordered pairs
results = run_lagged_stock_search(
    df_iter,
    lag_days=lag_days,
    lookback_days=lookback_days,
    persistence_window_days=persistence_window,
    add_significance=add_significance,
    min_significance=min_significance,
    fdr_q=fdr_q,
    sector_df=None,
    market_df=market_df,
    adjust_returns=adjust_returns,
    debug=False,
)
print("   [OK] Lead-lag analysis complete")

In [ ]:
# ============================================================================
# High-quality pair query (uses the SQL in the correlation library cell)
# ============================================================================

pairs_pd = select_high_quality_pairs(
    results,
    min_significance=min_significance,
    min_correlation=min_correlation,
    min_sign_fraction=min_sign_fraction,
    recent_days=recent_days,
    result_limit=result_limit,
    lag_days=lag_days,
)

print(f"[OK] {len(pairs_pd):,} unique high-quality pairs")

In [ ]:
# ============================================================================
# Upload results to S3
# ============================================================================

from datetime import datetime

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

pairs_pd["run_timestamp"] = run_timestamp
pairs_pd["iteration"] = 1
pairs_pd["lag_days"] = lag_days
pairs_pd["lookback_days"] = lookback_days
pairs_pd["persistence_window"] = persistence_window

for col in pairs_pd.columns:
    if pd.api.types.is_datetime64_any_dtype(pairs_pd[col]):
        pairs_pd[col] = pairs_pd[col].astype(str)

file_key = f"{cfg.correlation_prefix}/{run_timestamp}/data.parquet"
upload_parquet(pairs_pd, s3, cfg.s3_bucket, file_key)
print(f"Uploaded -> s3://{cfg.s3_bucket}/{file_key}")
print(f"Runtime: {timedelta(seconds=int(time.time() - start_time))}")